# Este archivo contiene la limpieza de los datos completos

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [4]:
# Config rutas

PATH = Path('/home/jonathan.aragonportillo/tensorflow/TS-Dengue')
PATH = PATH.parent if PATH.name=="notebooks" else PATH

print(f"La nueva ruta de los archivos es: {PATH} \n")
os.chdir(PATH)

In [7]:
# Data

df = pd.read_csv(PATH / 'data/raw/full_data.csv')

df.head()

In [9]:
print(list(df.columns))

In [10]:
df.shape

## Proceso de análisis descriptivos

In [13]:
df[['year', 'week', 'canton']] = df['week_canton'].str.split('-', expand=True).astype(int)

semanas_por_canton = df.groupby('canton')['week_canton'].nunique()
print("Semanas por cantón (min, max):", semanas_por_canton.min(), semanas_por_canton.max())

## Análisis de Valores faltantes

In [22]:
from seaborn import heatmap
import matplotlib.pyplot as plt

In [19]:
select_col = [col for col in df.columns if '_lag_' not in col]

In [20]:
df = df[select_col]

No hay valores faltantes.

In [23]:
heatmap(df.isna(), cbar=False)
plt.show()

## Analizando la cantidad de ceros que existen


La cantidad de ceros reportados es muy grande, representa el 60% aprox, entonces no se puede usar MSE.

In [28]:
sum(df['casos'] == 0)/df.shape[0] * 100

In [29]:
np.var(df['casos'])/np.mean(df['casos'])

In [33]:
s = pd.to_numeric(df['casos'], errors='coerce')

descriptivo = pd.DataFrame([{
    'tipo': df['casos'].dtype,
    'pct_nulos': round(s.isna().mean() * 100, 2),
    'pct_ceros': round((s == 0).mean() * 100, 2),
    'media': round(s.mean(), 2),
    'std': round(s.std(ddof=1), 2),
    'min': round(s.min(), 2),
    'p50': round(s.median(), 2),
    'max': round(s.max(), 2)
}])

descriptivo